# Week 9 — BWT: compress and search

Open in VS Code. **Select Kernel** (Python 3) in the top right. Then **Run All** from the top.

The code already runs. Read the comments, change one input if you want, watch the printout. Then answer the questions.

DNA example: $T=\mathtt{ACTGGAC\$}$ → $\mathrm{BWT}=\mathtt{CG\$AAGTC}$. Ranks and search: $T=\mathtt{abracadabra\$}$.

In [ ]:
def rotations(T):
    return [T[i:] + T[:i] for i in range(len(T))]


T = "ACTGGAC$"
rots = rotations(T)
rots


## BWT = last column of the sorted rotations


In [ ]:
sorted_rots = sorted(rots)
for r in sorted_rots:
    print(r)

bwt = "".join(r[-1] for r in sorted_rots)
print("BWT =", bwt)


## Same suffixes as week 8

$L$ is the letter immediately before each sorted suffix (wrap at the start).


In [ ]:
T2 = "ACA$"
pairs = sorted((T2[i:], i) for i in range(len(T2)))
L = "".join(T2[i - 1] for _, i in pairs)
for suf, i in pairs:
    print(suf, "  before", T2[i - 1])
print("BWT =", L)


## Invert by sorting

Prepend $L$ and sort, until rows have length $m$. The row that **ends** with `$` is $T$.


In [ ]:
def invert(L):
    rows = [""] * len(L)
    for _ in range(len(L)):
        rows = sorted(L[i] + rows[i] for i in range(len(L)))
    for r in rows:
        if r.endswith("$"):
            return r
    return None


print(invert(bwt))


## Invert by last-to-first

Start in the row whose $F$ is `$`. Read $L$, jump to the same letter in $F$ (first row of that letter in $F$ plus its B-rank in $L$), repeat. The letters come out backwards.

In [ ]:
def invert_lf(L):
    F = "".join(sorted(L))
    first = {c: F.index(c) for c in set(F)}
    row, letters = 0, []
    print(" row  L  B-rank  next row")
    while L[row] != "$":
        c = L[row]
        b = L[:row].count(c)          # copies of c above this row in L
        nxt = first[c] + b
        print(f"  {row}   {c}    {b}       {nxt}")
        letters.append(c)
        row = nxt
    return "".join(reversed(letters)) + "$"


print("T =", invert_lf("CG$AAGTC"))

## T-ranks vs B-ranks

$T=$ `abracadabra$`. T-rank = how many copies of that letter already appeared to the left in $T$. B-rank = how many copies sit above you in this column. They are not the same number.


In [ ]:
T3 = "abracadabra$"
seen = {}
print("T-ranks:")
for ch in T3:
    seen[ch] = seen.get(ch, -1) + 1
    print(f"  {ch}{seen[ch]}", end="")
print()

L3 = "".join(r[-1] for r in sorted(T3[i:] + T3[:i] for i in range(len(T3))))
F3 = "".join(sorted(L3))
print("F =", F3)
print("L =", L3)

def branks(col):
    c, out = {}, []
    for ch in col:
        c[ch] = c.get(ch, -1) + 1
        out.append(f"{ch}{c[ch]}")
    return out

print("B-ranks F", branks(F3))
print("B-ranks L", branks(L3))
print("first a in L is B-rank a0; T-rank a0 is the leftmost a in T — different letters.")


## Backward search

Walk the pattern last letter first. The matching rows stay a contiguous block.


In [ ]:
def index(T):
    rows = sorted((T[i:] + T[:i], i) for i in range(len(T)))
    F = "".join(rot[0] for rot, _ in rows)
    L = "".join(rot[-1] for rot, _ in rows)
    starts = [i for _, i in rows]
    return F, L, starts


def show(F, L, lo, hi, label):
    caret = "".join("^" if lo <= i < hi else " " for i in range(len(F)))
    print(f"  {label:>8}  F {F}")
    print(f"            L {L}")
    print(f"              {caret}  [{lo}, {hi})")


def search(T, p):
    F, L, starts = index(T)
    print(f"T = {T}    search {p!r} (last letter first)\n")
    if p[-1] not in F:
        print("  empty — last letter does not occur")
        return []
    lo = F.find(p[-1])
    hi = lo + F.count(p[-1])
    show(F, L, lo, hi, p[-1:])
    for k in range(len(p) - 2, -1, -1):
        c = p[k]
        ranks = [L[:i].count(c) for i in range(lo, hi) if L[i] == c]
        if not ranks:
            print(f"  {p[k:]:>8}  empty — no hit")
            return []
        lo = F.find(c) + ranks[0]
        hi = F.find(c) + ranks[-1] + 1
        show(F, L, lo, hi, p[k:])
    hits = sorted(starts[lo:hi])
    print(f"\nhits at {hits}")
    return hits


search("abracadabra$", "abra")
print()
search("abracadabra$", "acb")
# search("abracadabra$", "cad")
# search("ACTGGAC$", "AC")


## Build your own search

The search above prints each block. Now write it yourself, in five functions. Fill in each function in the next cell, one step at a time, and re-run the test cell after each step.

1. **Transform.** Sort the rotations of $T$. Keep $L$, and the text position where each row starts.
2. **Where each letter starts in $F$.** For each letter $c$: $F_c$ = number of letters in $L$ smaller than $c$.
3. **Counts above each row.** For each letter $c$ and each row $r$: $\mathrm{occ}_c(r)$ = copies of $c$ in $L$ above row $r$, for $r = 0, \dots, n$.
4. **Walk the pattern backward.** Start with all rows, $lo = 0$, $hi = n$. For each letter $c$ of $p$, last to first: $lo = F_c + \mathrm{occ}_c(lo)$ and $hi = F_c + \mathrm{occ}_c(hi)$. Stop if the block is empty.
5. **Report.** The rows $lo, \dots, hi-1$ start at the text positions where $p$ occurs.

For $T = \mathtt{abracadabra\$}$ you should get: `L = "ard$rcaaaabb"`, `F_b = 6`, $\mathrm{occ}_{\mathtt{a}} = [0,1,1,1,1,1,1,2,3,4,5,5,5]$, and `abra` at `[0, 7]`. The slides "Implementing the search" walk through every step. The solution is the last cell of this notebook.

In [ ]:
def my_bwt(T):
    """Step 1. Return (L, rows): L as a string, rows[r] = text position where row r starts."""
    # hint: sort range(len(T)) with key=lambda i: T[i:] + T[:i]; the L letter of row r is T[rows[r] - 1]
    raise NotImplementedError


def my_first_rows(L):
    """Step 2. Return a dict {letter: first row of that letter in F}."""
    # hint: go through sorted(set(L)) and keep a running total of L.count(c)
    raise NotImplementedError


def my_occ_table(L):
    """Step 3. Return a dict {letter: list of n + 1 counts}; occ[c][r] = copies of c in L[:r]."""
    raise NotImplementedError


def my_backward_search(p, first, occ, n):
    """Step 4. Return the block (lo, hi) of rows that begin with p; (0, 0) if p does not occur."""
    raise NotImplementedError


def my_find(T, p):
    """Step 5. Return the sorted text positions where p occurs in T."""
    raise NotImplementedError

In [ ]:
import random

T = "abracadabra$"
checks = [
    ("step 1", lambda: my_bwt(T) == ("ard$rcaaaabb", [11, 10, 7, 0, 3, 5, 8, 1, 4, 6, 9, 2])),
    ("step 2", lambda: my_first_rows("ard$rcaaaabb") == {"$": 0, "a": 1, "b": 6, "c": 8, "d": 9, "r": 10}),
    ("step 3", lambda: my_occ_table("ard$rcaaaabb")["a"] == [0, 1, 1, 1, 1, 1, 1, 2, 3, 4, 5, 5, 5]),
    ("step 4", lambda: my_backward_search("abra", my_first_rows("ard$rcaaaabb"),
                                          my_occ_table("ard$rcaaaabb"), 12) == (2, 4)),
    ("step 5", lambda: my_find(T, "abra") == [0, 7] and my_find(T, "acb") == []),
]
for name, check in checks:
    try:
        print(name, "ok" if check() else "WRONG ANSWER")
    except NotImplementedError:
        print(name, "not written yet")
        break


def scan(T, p):
    """The slow reference: try every start position."""
    return [i for i in range(len(T)) if T.startswith(p, i)]


try:
    for _ in range(300):
        t = "".join(random.choice("ACGT") for _ in range(30)) + "$"
        p = "".join(random.choice("ACGT") for _ in range(3))
        assert my_find(t, p) == scan(t, p), (t, p)
    print("300 random texts: same answers as the plain scan")
except NotImplementedError:
    pass

## Haplotypes: the positional transform

Six haplotypes, ten sites. The last one is a recombinant. At each site, put the haplotypes with allele 0 first and those with allele 1 after, each group in its old order. Then look at the recombinant's neighbours: the one sharing the longest stretch that ends at this site is its current copying partner.

In [ ]:
haps = ["0010010111", "1000011111", "1111010101", "1111101011", "1000011101", "0010001011"]
r = 5                                     # the recombinant


def shared_back(i, j, k):
    """How many sites, ending at site k, haplotypes i and j agree on."""
    n = 0
    while k - n >= 0 and haps[i][k - n] == haps[j][k - n]:
        n += 1
    return n


order = list(range(len(haps)))
for k in range(len(haps[0])):
    order = [i for i in order if haps[i][k] == "0"] + [i for i in order if haps[i][k] == "1"]
    pos = order.index(r)
    neighbours = [order[p] for p in (pos - 1, pos + 1) if 0 <= p < len(order)]
    best = max(neighbours, key=lambda j: shared_back(r, j, k))
    print(f"site {k}: order {order}   partner {best}, shared for {shared_back(r, best, k)} sites")

## Questions

Without looking at notes. **Q1–Q2:** today’s idea. **Q3:** a situation we did not practise.

**Q1.** What do the T-rank and the B-rank of a letter count? Why does the last-to-first jump only need B-ranks?

A1.

**Q2.** Why does the search read the pattern from its last letter to its first? What does an empty block mean?

A2.

**Q3.** A read differs from the genome by one sequencing error, so the backward search ends with an empty block. At which step does it become empty, and how could the search be changed to still find the read's position?

A3.

## Solution: build your own search

Open this only after trying the five steps yourself.

In [ ]:
# Solution: the five steps as on the slides. Try your own first.

def bwt(T):
    rows = sorted(range(len(T)), key=lambda i: T[i:] + T[:i])
    L = "".join(T[i - 1] for i in rows)
    return L, rows


def first_rows(L):
    first, total = {}, 0
    for c in sorted(set(L)):
        first[c] = total
        total += L.count(c)
    return first


def occ_table(L):
    occ = {c: [0] for c in set(L)}
    for letter in L:
        for c in occ:
            occ[c].append(occ[c][-1] + (letter == c))
    return occ


def backward_search(p, first, occ, n):
    lo, hi = 0, n
    for c in reversed(p):
        if c not in first:
            return 0, 0
        lo = first[c] + occ[c][lo]
        hi = first[c] + occ[c][hi]
        if lo >= hi:
            return 0, 0
    return lo, hi


def find(T, p):
    L, rows = bwt(T)
    lo, hi = backward_search(p, first_rows(L), occ_table(L), len(L))
    return sorted(rows[lo:hi])


for p in ["abra", "a", "cad", "acb"]:
    print(p, find("abracadabra$", p))